# Ñ contra 16 bibliotecas de otros lenguajes, en las T4 de Kaggle

Las mismas seis pruebas (`matmul` 1024 y 2048, `seno`, `x*2+1` y `suma` de 2^24 números, `softmax` 4096×1024 y `fft` de 2^20) con **los mismos datos** en Ñ (C1EÑ en jit, nativo y GPU) y en:

| lenguaje | biblioteca | aquí | por defecto |
|---|---|---|---|
| Python | PyTorch | CPU y GPU | **activo** |
| C++ | Eigen | CPU | **activo** |
| C | OpenBLAS | CPU (BLAS solo hace matmul; el resto, bucles C con OpenMP) | **activo** |
| JavaScript | TensorFlow.js (tfjs-node) | CPU, **float32** | **activo** |
| Go | Gorgonia | CPU | **activo** |
| Fortran | Intel MKL | CPU | **activo** |
| Julia | CUDA.jl | CPU y GPU | apagado (instalar tarda ~10–15 min) |
| Rust | Candle | CPU y GPU | apagado (compilar con CUDA ~15–25 min) |
| R | torch | CPU y GPU | apagado (~2 GB) |
| Java | ND4J | CPU | apagado |
| MATLAB | **Octave en su lugar** (MATLAB necesita licencia) | CPU | apagado |
| C# | TorchSharp | CPU y GPU | apagado (~2,5 GB) |
| Swift | **Swift sin Metal** (Metal solo existe en Apple) | CPU, un hilo | apagado |
| Scala | Breeze | CPU | apagado |
| Kotlin | KotlinDL (se mide TensorFlow 1.15 para Java, su motor) | CPU | apagado |

Cada uno escribe `RES,prueba,tamaño,dispositivo,ms,huella,precisión`. La **huella** (suma del resultado) se compara con la de NumPy en doble precisión: si un lenguaje da otra cosa, la tabla lo marca. Si uno falla al instalarse o al correr, se salta, se apunta por qué y el resto sigue.

**Por qué algunos solo en CPU:** Eigen, OpenBLAS, MKL, Gorgonia y Breeze son de CPU. TensorFlow.js para GPU (tfjs-node-gpu), ND4J para CUDA y TensorFlow 1.15 (KotlinDL) piden CUDA 10/11, y Kaggle tiene CUDA 12. MATLAB y Metal no se pueden instalar aquí: sus sustitutos están marcados como tales y nunca se hacen pasar por el original.

**Antes de empezar:** *Settings → Accelerator → GPU T4 x2* e *Internet → On*. Con lo activo por defecto tarda unos 15–20 minutos; con todo, una hora o más.

In [ ]:
# 1 · Qué corre. Cambia True/False y vuelve a correr desde aquí.
ESCALA = 'normal'            # 'chica' para probar que todo corre en pocos minutos
ACTIVOS = {
    'Ñ': True,
    'PyTorch': True, 'Eigen': True, 'OpenBLAS': True, 'TensorFlow.js': True, 'Gorgonia': True, 'MKL': True,
    'CUDA.jl': False, 'Candle': False, 'torch (R)': False, 'ND4J': False, 'Octave (en lugar de MATLAB)': False,
    'TorchSharp': False, 'Swift (sin Metal)': False, 'Breeze': False, 'KotlinDL': False,
}
TRABAJO = '/kaggle/working'

In [ ]:
# 2 · Node 22 y el proyecto (dataset subido, o git clone)
import os, subprocess, shutil, re, glob, time, json
def version_node():
    try: return subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError: return ''
v = version_node()
if not v or int(re.match(r'v(\d+)', v).group(1)) < 18:
    NODE = 'v22.12.0'
    subprocess.run(f'curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz | tar -xJ -C {TRABAJO}', shell=True, check=True)
    os.environ['PATH'] = f'{TRABAJO}/node-{NODE}-linux-x64/bin:' + os.environ['PATH']
print('node', version_node())
PROYECTO = f'{TRABAJO}/n'
candidatos = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/motor/rutas.js', recursive=True)]
if os.path.exists(PROYECTO): print('ya estaba en', PROYECTO)
elif candidatos:
    shutil.copytree(candidatos[0], PROYECTO, ignore=shutil.ignore_patterns('node_modules')); print('copiado desde', candidatos[0])
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/diegovelandiabarajas1-lang/Espanol-Like', PROYECTO], check=True)
os.chdir(PROYECTO)
COMP = f'{PROYECTO}/c1en/comparar'
if not os.path.exists(f'{COMP}/n.esl'):
    commit = subprocess.run(['git', 'log', '-1', '--format=%h %ad %s', '--date=short'], capture_output=True, text=True).stdout.strip()
    raise SystemExit(f'falta c1en/comparar/ (los programas de la comparación). El repositorio está en «{commit}»: haz git push desde tu PC.')
print(subprocess.run(['git', 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv
!lscpu | grep -E '^(Model name|CPU\(s\))'

In [ ]:
# 3 · Ayudantes: instalar, correr un programa y quedarse con sus líneas RES
import pandas as pd
ARG = ['chica'] if ESCALA == 'chica' else []
filas, estado = [], {}
def sh(cmd, timeout=3600, env=None, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, cwd=cwd,
                       env=dict(os.environ, **(env or {})))
    if r.returncode:
        raise RuntimeError(f'«{cmd[:120]}» salió con {r.returncode}:\n' + '\n'.join((r.stdout + r.stderr).strip().splitlines()[-15:]))
    return r.stdout
def correr(nombre, lenguaje, biblioteca, cmd, timeout=3600, env=None, cwd=None):
    t0 = time.time()
    salida = sh(cmd, timeout=timeout, env=env, cwd=cwd)
    n = 0
    for l in salida.splitlines():
        if l.startswith('#'): print('   ', l)
        if not l.startswith('RES,'): continue
        _, prueba, tam, disp, ms, huella, prec = l.strip().split(',')
        filas.append(dict(entrada=nombre, lenguaje=lenguaje, biblioteca=biblioteca, prueba=prueba, tam=tam, dispositivo=disp,
                          ms=float(ms), huella=float(huella), precision=prec))
        n += 1
    print(f'   {n} medidas en {time.time() - t0:.0f} s')
    return n
def seccion(nombre, fn):
    if not ACTIVOS.get(nombre):
        estado[nombre] = 'apagado'; print(f'{nombre}: apagado (ACTIVOS)'); return
    t0 = time.time()
    try:
        print(f'── {nombre}'); n = fn()
        estado[nombre] = f'✓ {n} medidas · {time.time() - t0:.0f} s'
    except Exception as e:
        estado[nombre] = '✗ ' + str(e).splitlines()[0][:200]
        print(f'✗ {nombre} falló:\n{e}')
def apt(*paquetes):
    sh('apt-get update -qq', timeout=900)
    sh('DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends ' + ' '.join(paquetes), timeout=1800)
HAY_GPU = shutil.which('nvidia-smi') is not None and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
print('GPU:', HAY_GPU, '· escala', ESCALA)

## Ñ (C1EÑ, motor-exp): jit, nativo y GPU

In [ ]:
def n_():
    sh('npm install koffi --no-audit --no-fund --loglevel=error', timeout=900)
    sh('node motor-exp/util/construir.js')
    total = 0
    for d in ['jit', 'nativo'] + (['gpu'] if HAY_GPU else []):
        total += correr('Ñ · ' + d, 'Ñ', 'C1EÑ', f'node motor-exp/util/n-node.js c1en/comparar/n.esl {d} ' + ' '.join(ARG), timeout=3600)
    return total
seccion('Ñ', n_)

## Python · PyTorch (CPU y GPU)

In [ ]:
seccion('PyTorch', lambda: correr('Python · PyTorch', 'Python', 'PyTorch', f'python {COMP}/pytorch.py ' + ' '.join(ARG)))

## C++ · Eigen · C · OpenBLAS · Fortran · Intel MKL (CPU)

In [ ]:
def eigen_():
    apt('libeigen3-dev', 'g++')
    sh(f'g++ -O3 -march=native -fopenmp -I/usr/include/eigen3 {COMP}/eigen.cpp -o {TRABAJO}/eigen')
    return correr('C++ · Eigen', 'C++', 'Eigen', f'{TRABAJO}/eigen ' + ' '.join(ARG))
seccion('Eigen', eigen_)

def openblas_():
    apt('libopenblas-dev', 'gcc')
    sh(f'gcc -O3 -march=native -fopenmp {COMP}/openblas.c -o {TRABAJO}/openblas -lopenblas -lm')
    return correr('C · OpenBLAS', 'C', 'OpenBLAS', f'{TRABAJO}/openblas ' + ' '.join(ARG))
seccion('OpenBLAS', openblas_)

def mkl_():
    apt('gfortran')
    M = f'{TRABAJO}/mklenv'
    if not os.path.exists(f'{M}/bin/pip'): sh(f'python -m venv {M}')
    sh(f'{M}/bin/pip install -q mkl mkl-devel', timeout=1800)
    rt = sorted(glob.glob(f'{M}/lib/libmkl_rt.so.*'))
    if not rt: raise RuntimeError('no apareció libmkl_rt.so.* en el entorno de MKL')
    sh(f'gfortran -O3 -march=native -fopenmp -I{M}/include {M}/include/mkl_dfti.f90 {COMP}/mkl.f90 '
       f'-L{M}/lib -l:{os.path.basename(rt[-1])} -Wl,-rpath,{M}/lib -o {TRABAJO}/mklf', cwd=TRABAJO)
    return correr('Fortran · MKL', 'Fortran', 'Intel MKL', f'{TRABAJO}/mklf ' + ' '.join(ARG))
seccion('MKL', mkl_)

## JavaScript · TensorFlow.js · Go · Gorgonia (CPU)

In [ ]:
def tfjs_():
    D = f'{TRABAJO}/tfjs'; os.makedirs(D, exist_ok=True)
    sh('npm init -y >/dev/null && npm install @tensorflow/tfjs-node@4.22.0 --no-audit --no-fund --loglevel=error', cwd=D, timeout=1800)
    shutil.copy(f'{COMP}/tfjs.js', D)
    return correr('JavaScript · TensorFlow.js', 'JavaScript', 'TensorFlow.js', 'node tfjs.js ' + ' '.join(ARG), cwd=D)
seccion('TensorFlow.js', tfjs_)

def go_():
    if not os.path.exists(f'{TRABAJO}/go/bin/go'):
        ver = json.loads(sh('curl -fsSL "https://go.dev/dl/?mode=json"'))[0]['version']
        sh(f'curl -fsSL https://go.dev/dl/{ver}.linux-amd64.tar.gz | tar -xz -C {TRABAJO}', timeout=900)
    env = {'PATH': f'{TRABAJO}/go/bin:' + os.environ['PATH'], 'GOPATH': f'{TRABAJO}/gopath', 'GOCACHE': f'{TRABAJO}/gocache', 'GOFLAGS': '-mod=mod'}
    D = f'{TRABAJO}/gorgonia'; os.makedirs(D, exist_ok=True)
    shutil.copy(f'{COMP}/gorgonia.go', f'{D}/main.go')
    if not os.path.exists(f'{D}/go.mod'): sh('go mod init comparar', env=env, cwd=D)
    sh('go get gorgonia.org/tensor@v0.9.24 && go build -o gorgonia .', env=env, cwd=D, timeout=1800)
    return correr('Go · Gorgonia', 'Go', 'Gorgonia', './gorgonia ' + ' '.join(ARG), env=env, cwd=D)
seccion('Gorgonia', go_)

## Julia · CUDA.jl · Rust · Candle (CPU y GPU)

In [ ]:
def julia_():
    J = f'{TRABAJO}/julia-1.13.2/bin/julia'
    if not os.path.exists(J):
        sh(f'curl -fsSL https://julialang-s3.julialang.org/bin/linux/x64/1.13/julia-1.13.2-linux-x86_64.tar.gz | tar -xz -C {TRABAJO}', timeout=1800)
    env = {'JULIA_DEPOT_PATH': f'{TRABAJO}/julia-depot', 'JULIA_PKG_PRECOMPILE_AUTO': '1'}
    sh(f'{J} -e \'using Pkg; Pkg.add(["CUDA", "FFTW"]); using CUDA, FFTW\'', env=env, timeout=3600)
    if not HAY_GPU: env['SIN_GPU'] = '1'
    return correr('Julia · CUDA.jl', 'Julia', 'CUDA.jl', f'{J} --threads=auto {COMP}/cudajl.jl ' + ' '.join(ARG), env=env, timeout=3600)
seccion('CUDA.jl', julia_)

def candle_():
    cargo = os.path.expanduser('~/.cargo/bin/cargo')
    if not os.path.exists(cargo): sh('curl -fsSL https://sh.rustup.rs | sh -s -- -y --profile minimal', timeout=1800)
    nvcc = shutil.which('nvcc') or ('/usr/local/cuda/bin/nvcc' if os.path.exists('/usr/local/cuda/bin/nvcc') else None)
    env = {'PATH': os.path.dirname(cargo) + ':/usr/local/cuda/bin:' + os.environ['PATH'], 'CUDA_COMPUTE_CAP': '75',
           'CARGO_TARGET_DIR': f'{TRABAJO}/candle-target'}
    D = f'{TRABAJO}/candle'
    if os.path.exists(D): shutil.rmtree(D)
    shutil.copytree(f'{COMP}/candle', D)
    rasgos = '--features cuda' if (HAY_GPU and nvcc) else ''
    if HAY_GPU and not nvcc: print('   sin nvcc: Candle solo en CPU')
    sh(f'cargo build --release {rasgos}', env=env, cwd=D, timeout=3600)
    return correr('Rust · Candle', 'Rust', 'Candle', f'{TRABAJO}/candle-target/release/comparar-candle ' + ' '.join(ARG), env=env)
seccion('Candle', candle_)

## R · torch · C# · TorchSharp (CPU y GPU)

In [ ]:
def rtorch_():
    if not shutil.which('Rscript'):
        apt('software-properties-common', 'dirmngr', 'wget', 'gnupg')
        sh('wget -qO- https://cloud.r-project.org/bin/linux/ubuntu/marutter_pubkey.asc > /etc/apt/trusted.gpg.d/cran_ubuntu_key.asc')
        sh('add-apt-repository -y "deb https://cloud.r-project.org/bin/linux/ubuntu $(lsb_release -cs)-cran40/"', timeout=900)
        apt('r-base-core', 'r-base-dev')
    tipo = 'cu128' if HAY_GPU else 'cpu'
    instalar = ('options(timeout = 1800); '
                'ver <- available.packages(repos = "https://cloud.r-project.org")["torch", "Version"]; '
                f'options(repos = c(torch = sprintf("https://torch-cdn.mlverse.org/packages/{tipo}/%s/", ver), CRAN = "https://cloud.r-project.org")); '
                'install.packages("torch"); library(torch); if (!torch_is_installed()) install_torch()')
    sh(f"Rscript -e '{instalar}'", env={'TORCH_INSTALL': '1'}, timeout=3600)
    return correr('R · torch', 'R', 'torch', f'Rscript {COMP}/rtorch.R ' + ' '.join(ARG), timeout=3600)
seccion('torch (R)', rtorch_)

def torchsharp_():
    DN = f'{TRABAJO}/dotnet'
    if not os.path.exists(f'{DN}/dotnet'):
        sh(f'curl -fsSL https://dot.net/v1/dotnet-install.sh -o {TRABAJO}/dotnet-install.sh && bash {TRABAJO}/dotnet-install.sh --channel 8.0 --install-dir {DN}', timeout=1800)
    env = {'DOTNET_ROOT': DN, 'PATH': DN + ':' + os.environ['PATH'], 'DOTNET_CLI_TELEMETRY_OPTOUT': '1', 'DOTNET_NOLOGO': '1',
           'NUGET_PACKAGES': f'{TRABAJO}/nuget'}
    D = f'{TRABAJO}/torchsharp'
    if os.path.exists(D): shutil.rmtree(D)
    shutil.copytree(f'{COMP}/torchsharp', D)
    paquete = 'TorchSharp-cuda-linux' if HAY_GPU else 'TorchSharp-cpu'
    sh(f'dotnet add package {paquete}', env=env, cwd=D, timeout=3600)
    sh('dotnet build -c Release -o salida', env=env, cwd=D, timeout=1800)
    return correr('C# · TorchSharp', 'C#', 'TorchSharp', 'dotnet salida/comparar.dll ' + ' '.join(ARG), env=env, cwd=D, timeout=3600)
seccion('TorchSharp', torchsharp_)

## Java · ND4J · Scala · Breeze · Kotlin · KotlinDL (CPU, en la JVM)

Los tres comparten un JDK y [Coursier](https://get-coursier.io) para bajar las bibliotecas de Maven Central.

In [ ]:
def jvm():
    if not shutil.which('javac'): apt('openjdk-17-jdk-headless')
    cs = f'{TRABAJO}/cs'
    if not os.path.exists(cs):
        sh(f'curl -fsSL https://github.com/coursier/launchers/raw/master/cs-x86_64-pc-linux.gz | gzip -d > {cs} && chmod +x {cs}', timeout=900)
    return cs

def nd4j_():
    cs = jvm()
    cp = sh(f'{cs} fetch --classpath org.nd4j:nd4j-native:1.0.0-M2.1 org.nd4j:nd4j-native:1.0.0-M2.1,classifier=linux-x86_64', timeout=1800).strip()
    return correr('Java · ND4J', 'Java', 'ND4J', f'java -Xmx12g -cp "{cp}" {COMP}/Nd4jBench.java ' + ' '.join(ARG), timeout=3600)
seccion('ND4J', nd4j_)

def breeze_():
    jvm(); apt('libopenblas-dev')                  # la libblas.so.3 que encuentra netlib
    sc = f'{TRABAJO}/scala-cli'
    if not os.path.exists(sc):
        sh(f'curl -fsSL https://github.com/VirtusLab/scala-cli/releases/latest/download/scala-cli-x86_64-pc-linux.gz | gzip -d > {sc} && chmod +x {sc}', timeout=900)
    return correr('Scala · Breeze', 'Scala', 'Breeze', f'{sc} run --server=false {COMP}/breeze.sc -- ' + ' '.join(ARG), timeout=3600)
seccion('Breeze', breeze_)

def kotlindl_():
    cs = jvm()
    K = f'{TRABAJO}/kotlinc/bin/kotlinc'
    if not os.path.exists(K):
        rel = json.loads(sh('curl -fsSL https://api.github.com/repos/JetBrains/kotlin/releases/latest'))
        url = next(a['browser_download_url'] for a in rel['assets'] if re.match(r'kotlin-compiler-[\d.]+\.zip$', a['name']))
        sh(f'curl -fsSL -o {TRABAJO}/kotlinc.zip {url} && cd {TRABAJO} && python -m zipfile -e kotlinc.zip . && chmod +x kotlinc/bin/*', timeout=1800)
    cp = sh(f'{cs} fetch --classpath org.jetbrains.kotlinx:kotlin-deeplearning-tensorflow:0.5.2', timeout=1800).strip()
    sh(f'{K} {COMP}/kotlindl.kt -cp "{cp}" -include-runtime -d {TRABAJO}/kotlindl.jar', timeout=1800)
    return correr('Kotlin · KotlinDL', 'Kotlin', 'KotlinDL', f'java -Xmx12g -cp "{TRABAJO}/kotlindl.jar:{cp}" KotlindlKt ' + ' '.join(ARG), timeout=3600)
seccion('KotlinDL', kotlindl_)

## Los sustitutos: Octave en lugar de MATLAB · Swift sin Metal

- **MATLAB** necesita licencia y no se instala en Kaggle. **GNU Octave** corre el mismo archivo `.m` (que también corre en MATLAB tal cual), pero los tiempos son los de Octave (OpenBLAS y FFTW), no los de MATLAB, y sin Parallel Computing Toolbox: solo CPU.
- **Metal Performance Shaders** solo existe en Mac, iPhone y iPad. En Linux tampoco hay Accelerate. Se mide **Swift puro** (bucles a mano, un hilo): es Swift, no MPS, y así sale en la tabla.

In [ ]:
def octave_():
    apt('octave', 'libopenblas0')
    return correr('MATLAB → Octave', 'MATLAB (Octave)', 'Octave', f'octave-cli --no-gui {COMP}/octave.m ' + ' '.join(ARG), timeout=3600)
seccion('Octave (en lugar de MATLAB)', octave_)

def swift_():
    S = os.path.expanduser('~/.local/share/swiftly')
    if not shutil.which('swiftc') and not os.path.exists(f'{S}/bin/swiftc'):
        apt('binutils', 'git', 'gnupg2', 'libc6-dev', 'libcurl4-openssl-dev', 'libedit2', 'libpython3-dev', 'libsqlite3-0',
            'libxml2-dev', 'libz3-dev', 'pkg-config', 'tzdata', 'unzip', 'zlib1g-dev', 'libstdc++-12-dev', 'libgcc-12-dev')
        sh(f'cd {TRABAJO} && curl -fsSLO https://download.swift.org/swiftly/linux/swiftly-$(uname -m).tar.gz && tar zxf swiftly-$(uname -m).tar.gz '
           f'&& ./swiftly init --assume-yes --quiet-shell-followup', timeout=3600)
    env = {'PATH': f'{S}/bin:' + os.environ['PATH']}
    sh(f'swiftc -Ounchecked {COMP}/swift.swift -o {TRABAJO}/swiftbench', env=env, timeout=1800)
    return correr('Swift (sin Metal)', 'Swift', 'Swift puro, sin Metal', f'{TRABAJO}/swiftbench ' + ' '.join(ARG), env=env, timeout=3600)
seccion('Swift (sin Metal)', swift_)

## ¿Calculan lo mismo? La huella contra NumPy

Cada huella se compara con la de NumPy en doble precisión (`c1en/comparar/referencia.py`). En doble precisión la diferencia relativa tiene que ser pequeñísima (≤ 1e-9; las sumas de 16 millones de números, en distinto orden, se quedan en ~1e-12); en float32 (TensorFlow.js, la FFT de KotlinDL) ~1e-4.

In [ ]:
import importlib.util, numpy as np
spec = importlib.util.spec_from_file_location('referencia', f'{COMP}/referencia.py'); ref = importlib.util.module_from_spec(spec); spec.loader.exec_module(ref)
REF = ref.huellas((128, 256), 65536, 256, 1024, 16384) if ESCALA == 'chica' else ref.huellas()
df = pd.DataFrame(filas)
df['referencia'] = [REF.get((p, t), np.nan) for p, t in zip(df['prueba'], df['tam'])]
df['dif_rel'] = (df['huella'] - df['referencia']).abs() / df['referencia'].abs()
df['bien'] = np.where(df['precision'] == 'f32', df['dif_rel'] < 1e-3, df['dif_rel'] < 1e-9)
malos = df[~df['bien']]
print(f'{len(df)} medidas · {int(df["bien"].sum())} con la huella de NumPy · {len(malos)} distintas')
if len(malos): print(malos[['entrada', 'prueba', 'tam', 'dispositivo', 'huella', 'referencia', 'dif_rel']].to_string(index=False))
print('\nEstado de cada lenguaje:')
for k, v in estado.items(): print(f'  {k:30s} {v}')

## La tabla

Milisegundos, la mejor de 3 (menos es mejor). Cada columna es *lenguaje · biblioteca (dispositivo)*. Debajo, para cada prueba, quién va primero y cuántas veces más lento que el más rápido va cada uno, y la posición de Ñ.

In [ ]:
df['col'] = df['entrada'].str.replace('Ñ · ', 'Ñ ', regex=False) + df.apply(lambda r: '' if r['lenguaje'] == 'Ñ' else f" ({r['dispositivo']})", axis=1)
orden_tam = lambda s: s.map(lambda t: float(str(t).split('x')[0]))
ancho = df.pivot_table(index=['prueba', 'tam'], columns='col', values='ms', aggfunc='min')
ancho = ancho.reset_index().sort_values(['prueba', 'tam'], key=lambda s: orden_tam(s) if s.name == 'tam' else s).set_index(['prueba', 'tam'])
pd.set_option('display.width', 400); pd.set_option('display.max_columns', 60)
print(ancho.round(2).T.to_string())
print()
for (p, t), g in df.groupby(['prueba', 'tam'], sort=False):
    g = g.sort_values('ms'); mejor = g['ms'].iloc[0]
    linea = ' · '.join(f"{r['col']} {r['ms']:.1f}" + ('' if k == 0 else f" (×{r['ms'] / mejor:.1f})") for k, (_, r) in enumerate(g.iterrows()))
    pos = [k + 1 for k, (_, r) in enumerate(g.iterrows()) if r['lenguaje'] == 'Ñ']
    print(f'{p} {t}: Ñ en el puesto {pos} de {len(g)}\n   {linea}\n')

In [ ]:
import matplotlib.pyplot as plt
pruebas = list(dict.fromkeys(zip(df['prueba'], df['tam'])))
fig, ejes = plt.subplots(len(pruebas), 1, figsize=(14, 3.2 * len(pruebas)))
ejes = np.atleast_1d(ejes)
for ax, (p, t) in zip(ejes, pruebas):
    g = df[(df['prueba'] == p) & (df['tam'] == t)].sort_values('ms')
    colores = ['#c0392b' if l == 'Ñ' else ('#2e86c1' if d == 'gpu' else '#7f8c8d') for l, d in zip(g['lenguaje'], g['dispositivo'])]
    ax.barh(g['col'], g['ms'], color=colores); ax.set_xscale('log'); ax.invert_yaxis()
    ax.set_title(f'{p} · {t}  (ms, escala log; rojo Ñ, azul GPU, gris CPU)', fontsize=10); ax.tick_params(axis='y', labelsize=8)
plt.tight_layout(); plt.show()

mm = df[df['prueba'] == 'matmul'].copy()
mm['GFLOP/s'] = 2 * mm['tam'].astype(float) ** 3 / (mm['ms'] / 1000) / 1e9
print(mm.pivot_table(index='col', columns='tam', values='GFLOP/s').sort_values(mm['tam'].max(), ascending=False).round(1).to_string())

In [ ]:
# Los CSV para descargar (Output) — y para mandármelos
df.drop(columns=['col']).to_csv(f'{TRABAJO}/comparar-lenguajes.csv', index=False)
ancho.to_csv(f'{TRABAJO}/comparar-lenguajes-resumen.csv')
json.dump(dict(escala=ESCALA, estado=estado, commit=subprocess.run(['git', 'log', '-1', '--format=%h'], capture_output=True, text=True).stdout.strip()),
          open(f'{TRABAJO}/comparar-lenguajes-info.json', 'w'), ensure_ascii=False, indent=1)
for f in ['comparar-lenguajes.csv', 'comparar-lenguajes-resumen.csv', 'comparar-lenguajes-info.json']:
    print(f'{TRABAJO}/{f}: {os.path.getsize(f"{TRABAJO}/{f}") / 1024:.1f} KB')

## Cómo leerlo

- **Lo justo de la comparación:** todos en doble precisión salvo TensorFlow.js (float32) y la FFT de KotlinDL; todos con los datos ya en su dispositivo y el resultado sin copiar. PyTorch, R torch y TorchSharp usan cuBLAS y cuFFT en la GPU (con FMA); C1EÑ no usa FMA para dar los mismos bits que la CPU, así que en `matmul` de GPU su techo es la mitad.
- **CPU:** Eigen, OpenBLAS, MKL y PyTorch usan AVX/FMA y todos los núcleos; los de la JVM y Go dependen de su BLAS; Swift va en un hilo a propósito (es Swift puro).
- **Lo que no está:** OpenBLAS, Gorgonia, Candle, ND4J y Swift no traen FFT; esa casilla sale vacía.
- Si un lenguaje sale ✗ en el estado, su error está en su celda: mándamelo y lo arreglo (los programas de C++, C, Fortran, Octave y Ñ se probaron antes de entregarlos, y el de TensorFlow.js con su motor de JavaScript puro; los demás se prueban aquí por primera vez).